# kbprojection Exercises and Assignments

A beginner-friendly guided workbook for learning how to use `kbprojection` to run experiments.

This notebook is **offline-first**. It uses tiny in-notebook data, a mock LLM, and a mock LangPro prover so you can learn the experiment flow without API keys or a working LangPro endpoint. The final exercise shows how to diagnose the currently stale JSON LangPro endpoint, but it is disabled by default.

How to use this notebook:

1. Read the learning goal and assignment prompt.
2. Try the `TODO` starter cell.
3. Open the hint if you get stuck.
4. Compare with the tagged `Solution` cell.
5. Run the check cell.

Solution cells are tagged with `solution` and start with a `# Solution` comment.


## 0. Environment and imports

Run this first. It imports the local repository and shared objects used throughout the exercises.


In [5]:
import csv
import json
import random
import re
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

if sys.version_info < (3, 10):
    raise RuntimeError("Use a Python 3.10+ kernel. This repo uses modern type hints such as str | dict.")

# Prefer the local repository next to this notebook.
sys.path.insert(0, str(Path.cwd()))

import kbprojection
import kbprojection.filtering as filtering
import kbprojection.orchestration as orchestration
from kbprojection.loaders.base import DatasetLoader
from kbprojection.llm import extract_kb_from_output
from kbprojection.models import (
    ExperimentResult,
    ExperimentStatus,
    LangProResult,
    NLILabel,
    NLIProblem,
    ProblemConfig,
    TestMode,
)
from kbprojection.orchestration import process_kb_examples, process_single_problem
from kbprojection.prompts import fill_prompt, get_prompt, list_prompts
from kbprojection.filtering import normalize_kb_args, parse_kb_injection, pipeline_filter_kb_injections

random.seed(7)

print("Python:", sys.version.split()[0])
print("kbprojection loaded from:", kbprojection.__file__)
print("Notebook directory:", Path.cwd())


Python: 3.11.13
kbprojection loaded from: /Users/jorrytdejong/Documents/RNL paper publication/kbprojection/__init__.py
Notebook directory: /Users/jorrytdejong/Documents/RNL paper publication


## Exercise 1 — Create the core experiment objects

**Learning goal:** Understand the three objects that appear in nearly every experiment: `NLIProblem`, `ProblemConfig`, and `ExperimentResult`.

**Assignment:** Create a manual NLI problem, a config that tests both raw and filtered KB, and an empty result object around the problem.


In [1]:
print('hello')

hello


In [8]:
# TODO Exercise 1:
# 1. Create an NLIProblem named ex1_problem.
ex1_problem = NLIProblem(
    id="manual-jorryt",
    premises=["I am Jorryt"],  # List of premise sentences
    hypothesis="Jorryt is Jorryt",
    gold_label=NLILabel.ENTAILMENT,
    dataset="manual",
    split="exercise"
)
# 2. Create a ProblemConfig named ex1_config.
ex1_config = ProblemConfig(
    llm_provider="openai",
    model="gpt-5-mini",
    prompt_style="icl",
    post_process=True,
    test_mode=TestMode.FULL,
    run_ablation=False,
    verbose=True,
)
ex1_result = ExperimentResult(problem=ex1_problem)
# 4. Print their most important fields.
pass


**Hint:** `NLIProblem` needs `id`, `premises`, `hypothesis`, `gold_label`, `dataset`, and `split`. `ProblemConfig(test_mode=TestMode.BOTH)` is a good default for experiments.


In [3]:
# Solution Exercise 1
ex1_problem = NLIProblem(
    id="manual-dog",
    premises=["A dog is running in a park."],
    hypothesis="An animal is moving outdoors.",
    gold_label=NLILabel.ENTAILMENT,
    dataset="manual",
    split="exercise",
)

ex1_config = ProblemConfig(
    llm_provider="mock",
    model="mock-model",
    prompt_style="icl",
    test_mode=TestMode.BOTH,
    run_ablation=False,
    verbose=False,
)

ex1_result = ExperimentResult(problem=ex1_problem)

print("Problem:")
pprint(ex1_problem.model_dump())
print("\nConfig:")
pprint(ex1_config.model_dump())
print("\nInitial result status:", ex1_result.final_status.value)


Problem:
{'dataset': 'manual',
 'gold_label': <NLILabel.ENTAILMENT: 'entailment'>,
 'hypothesis': 'An animal is moving outdoors.',
 'id': 'manual-dog',
 'original_data': None,
 'premises': ['A dog is running in a park.'],
 'split': 'exercise'}

Config:
{'llm_provider': 'mock',
 'model': 'mock-model',
 'post_process': True,
 'prompt_style': 'icl',
 'run_ablation': False,
 'test_mode': <TestMode.BOTH: 'both'>,
 'verbose': False}

Initial result status: unknown


In [9]:
# Check Exercise 1
assert ex1_problem.gold_label == NLILabel.ENTAILMENT
assert ex1_config.test_mode == TestMode.BOTH
assert ex1_result.problem.id == "manual-dog"
print("Exercise 1 check passed.")


AssertionError: 

## Exercise 2 — Build a tiny dataset loader

**Learning goal:** See how the repository expects dataset loaders to expose problems.

**Assignment:** Build `MiniNLILoader`, an in-memory loader with four examples: one fixable by KB, one already correct, one still wrong, and one where KB gets filtered away.


In [5]:
# TODO Exercise 2:
# Define MiniNLILoader as a subclass of DatasetLoader.
# It should provide iter_problems(), get_problem(), and random_problem().
pass


**Hint:** For this workbook, the loader can keep a list of dictionaries in memory. It does not need to download or write dataset files.


In [10]:
# Solution Exercise 2
class MiniNLILoader(DatasetLoader):
    """A tiny in-memory dataset loader for offline exercises."""

    def __init__(self):
        self.data_dir = Path("<in-memory>")
        self._rows = [
            {
                "id": "fixed-dog",
                "premises": ["A dog is running in a park."],
                "hypothesis": "An animal is moving outdoors.",
                "gold_label": "entailment",
            },
            {
                "id": "already-smile",
                "premises": ["A woman is smiling."],
                "hypothesis": "A woman is smiling.",
                "gold_label": "entailment",
            },
            {
                "id": "still-wrong-cat",
                "premises": ["A cat is sleeping on the sofa."],
                "hypothesis": "A dog is running outside.",
                "gold_label": "entailment",
            },
            {
                "id": "empty-filter-chef",
                "premises": ["A chef is slicing a tomato."],
                "hypothesis": "Someone is cutting a tomato.",
                "gold_label": "entailment",
            },
        ]
        self._problems = [self._parse_row(row, "dev") for row in self._rows]
        self._by_id = {problem.id: problem for problem in self._problems}

    def _download(self):
        return None

    def _get_splits(self):
        return ["dev"]

    def _get_file_path(self, split):
        return self.data_dir / f"mini_{split}.jsonl"

    def _iter_file(self, file_path):
        yield from self._rows

    def _parse_row(self, row, split):
        return NLIProblem(
            id=row["id"],
            premises=row["premises"],
            hypothesis=row["hypothesis"],
            gold_label=self._parse_label(row["gold_label"]),
            dataset="mini",
            split=split,
            original_data=row,
        )

    def load(self, splits=None):
        return None

    def iter_problems(self, split="dev", label_filter=None):
        for problem in self._problems:
            if label_filter and problem.gold_label.value not in label_filter and problem.gold_label not in label_filter:
                continue
            yield problem

    def get_problem(self, key, split="dev"):
        return self._by_id[str(key)]

    def random_problem(self, split="dev", label_filter=None, exclude_keys=None):
        exclude_keys = exclude_keys or set()
        for problem in self.iter_problems(split=split, label_filter=label_filter):
            if problem.id not in exclude_keys:
                return problem
        return None

mini_loader = MiniNLILoader()
mini_problems = list(mini_loader.iter_problems("dev"))

for problem in mini_problems:
    print(problem.id, "->", problem.gold_label.value, "|", problem.hypothesis)


fixed-dog -> entailment | An animal is moving outdoors.
already-smile -> entailment | A woman is smiling.
still-wrong-cat -> entailment | A dog is running outside.
empty-filter-chef -> entailment | Someone is cutting a tomato.


In [11]:
# Check Exercise 2
assert len(mini_problems) == 4
assert mini_loader.get_problem("fixed-dog").hypothesis == "An animal is moving outdoors."
assert mini_loader._get_splits() == ["dev"]
print("Exercise 2 check passed.")


Exercise 2 check passed.


## Exercise 3 — Iterate, sample, and filter examples

**Learning goal:** Practice the dataset operations that experiment loops depend on.

**Assignment:** List all problem IDs, select entailment examples, retrieve a specific example, and sample one non-excluded example.


In [8]:
# TODO Exercise 3:
# Use mini_loader.iter_problems(), get_problem(), and random_problem().
pass


**Hint:** `label_filter` can be a set like `{"entailment"}`. `exclude_keys` prevents sampling an ID again.


In [9]:
# Solution Exercise 3
all_ids = [problem.id for problem in mini_loader.iter_problems("dev")]
entailment_ids = [problem.id for problem in mini_loader.iter_problems("dev", label_filter={"entailment"})]
selected_problem = mini_loader.get_problem("fixed-dog")
sampled_problem = mini_loader.random_problem("dev", label_filter={"entailment"}, exclude_keys={"fixed-dog"})

print("All IDs:", all_ids)
print("Entailment IDs:", entailment_ids)
print("Selected:", selected_problem.id, selected_problem.premises, "=>", selected_problem.hypothesis)
print("Sampled after excluding fixed-dog:", sampled_problem.id)


All IDs: ['fixed-dog', 'already-smile', 'still-wrong-cat', 'empty-filter-chef']
Entailment IDs: ['fixed-dog', 'already-smile', 'still-wrong-cat', 'empty-filter-chef']
Selected: fixed-dog ['A dog is running in a park.'] => An animal is moving outdoors.
Sampled after excluding fixed-dog: already-smile


In [10]:
# Check Exercise 3
assert "fixed-dog" in all_ids
assert set(entailment_ids) == set(all_ids)
assert selected_problem.id == "fixed-dog"
assert sampled_problem.id != "fixed-dog"
print("Exercise 3 check passed.")


Exercise 3 check passed.


## Exercise 4 — Inspect and fill prompt templates

**Learning goal:** Understand how `kbprojection` turns a problem into an LLM prompt.

**Assignment:** List prompt styles, choose the `icl` prompt, fill it with the `fixed-dog` example, and inspect the final task portion.


In [11]:
# TODO Exercise 4:
# Use list_prompts(), get_prompt(), and fill_prompt().
pass


**Hint:** `fill_prompt("icl", problem.premises, problem.hypothesis)` returns a complete prompt string.


In [12]:
# Solution Exercise 4
available_prompts = list_prompts()
raw_icl_template = get_prompt("icl")
filled_icl_prompt = fill_prompt("icl", selected_problem.premises, selected_problem.hypothesis)

print("Available prompts:", available_prompts)
print("\nTemplate starts with:")
print(raw_icl_template[:250])
print("\nFilled prompt ends with:")
print(filled_icl_prompt[-450:])


Available prompts: ['legacy_cot', 'legacy_least_to_most', 'legacy_icl', 'icl', 'cot']

Template starts with:
You are a knowledge base injection assistant. Your task is to generate semantic relations that bridge the meaning gap between a Premise and Hypothesis.

## STRICT RULES (Must follow exactly)

1. **Output format**: Your output MUST be enclosed in deli

Filled prompt ends with:
ART]
isa_wn(aim, draw)
[KB_END]
Note: "aim" and "draw" are NOT opposites - aiming can follow drawing. Use isa_wn if one action implies the other.

### Example 8: Compound noun hypernymy
Premise: A female swimmer exits the pool.
Hypothesis: A woman gets out of the pool.
[KB_START]
isa_wn(female swimmer, woman)
[KB_END]

## YOUR TASK

Premise: A dog is running in a park.
Hypothesis: An animal is moving outdoors.

Generate the knowledge injections:



In [13]:
# Check Exercise 4
assert "icl" in available_prompts
assert "A dog is running in a park." in filled_icl_prompt
assert "An animal is moving outdoors." in filled_icl_prompt
print("Exercise 4 check passed.")


Exercise 4 check passed.


## Exercise 5 — Parse mock LLM output

**Learning goal:** Learn how LLM text becomes a list of KB injection strings.

**Assignment:** Create a mock LLM response with `[KB_START]` and `[KB_END]`, then extract only valid `isa_wn` and `disj` lines.


In [14]:
# TODO Exercise 5:
# Create mock_llm_output and pass it to extract_kb_from_output().
pass


**Hint:** Invalid predicates such as `causes(...)` should not appear in the extracted output.


In [15]:
# Solution Exercise 5
mock_llm_output = """
Reasoning: dog is a kind of animal; running is a kind of moving.
[KB_START]
isa_wn(dog, animal)
isa_wn(run, move)
isa_wn(park, outdoors)
causes(run, tired)
[KB_END]
"""

extracted_kb = extract_kb_from_output(mock_llm_output)
print("Extracted KB:")
for relation in extracted_kb:
    print("-", relation)


Extracted KB:
- isa_wn(dog, animal)
- isa_wn(run, move)
- isa_wn(park, outdoors)


In [16]:
# Check Exercise 5
assert extracted_kb == ["isa_wn(dog, animal)", "isa_wn(run, move)", "isa_wn(park, outdoors)"]
print("Exercise 5 check passed.")


Exercise 5 check passed.


## Exercise 6 — Parse, normalize, and filter KB injections

**Learning goal:** See how raw KB strings are parsed, normalized, and filtered against the premise/hypothesis.

**Assignment:** Patch filtering with tiny offline tokenizer/lemmatizer helpers, then run `pipeline_filter_kb_injections` on the mock KB.


In [17]:
# TODO Exercise 6:
# 1. Parse one relation with parse_kb_injection().
# 2. Normalize one relation with normalize_kb_args().
# 3. Run pipeline_filter_kb_injections().
pass


**Hint:** The real package uses NLTK resources. The solution patches tiny offline replacements so this workbook remains runnable without downloads.


In [18]:
# Solution Exercise 6
class TinyLemma:
    SPECIAL = {
        "running": "run",
        "runs": "run",
        "ran": "run",
        "moving": "move",
        "moves": "move",
        "slicing": "slice",
        "slices": "slice",
        "cutting": "cut",
        "cuts": "cut",
        "drinking": "drink",
        "drunk": "drink",
    }

    def lemmatize(self, token, pos=None):
        token = token.lower()
        if token in self.SPECIAL:
            return self.SPECIAL[token]
        if pos == "v" and token.endswith("ing") and len(token) > 5:
            return token[:-3]
        if token.endswith("s") and len(token) > 3:
            return token[:-1]
        return token


def tiny_tokenize(text):
    return re.findall(r"[A-Za-z]+", text.lower())


def tiny_drop_leading_preposition(phrase):
    prepositions = {"in", "on", "at", "by", "with", "to", "from"}
    parts = phrase.split()
    if len(parts) == 3 and parts[0].lower() in prepositions:
        return " ".join(parts[1:])
    return phrase


def patch_filtering_for_offline_exercises():
    filtering.get_lemmatizer = lambda: TinyLemma()
    filtering.tokenize = tiny_tokenize
    filtering.drop_leading_preposition = tiny_drop_leading_preposition

patch_filtering_for_offline_exercises()

parsed_relation = parse_kb_injection("isa_wn(dog, animal)")
normalized_relation = normalize_kb_args("isa_wn(dog_house, animal)")
filtered_kb_results = pipeline_filter_kb_injections(
    extracted_kb,
    premise=selected_problem.premises,
    hypothesis=selected_problem.hypothesis,
    post_process=True,
    use_semantic=False,
)
filtered_kb = [str(result) for result in filtered_kb_results]

print("Parsed relation:", parsed_relation)
print("Normalized relation:", normalized_relation)
print("Filtered KB:", filtered_kb)


Parsed relation: ('isa_wn', 'dog', 'animal')
Normalized relation: isa_wn(dog house, animal)
Filtered KB: ['isa_wn(dog, animal)', 'isa_wn(run, move)', 'isa_wn(park, outdoors)', 'isa_wn(park, outdoor)']


In [19]:
# Check Exercise 6
assert parsed_relation == ("isa_wn", "dog", "animal")
assert normalized_relation == "isa_wn(dog house, animal)"
assert "isa_wn(dog, animal)" in filtered_kb
assert "isa_wn(run, move)" in filtered_kb
print("Exercise 6 check passed.")


Exercise 6 check passed.


## Exercise 7 — Write a mock LangPro prover

**Learning goal:** Understand what the real `langpro_api_call` returns without needing the live service.

**Assignment:** Implement `mock_langpro_api_call` so it returns `LangProResult` objects for baseline and KB-injected calls.


In [20]:
# TODO Exercise 7:
# Define mock_langpro_api_call(premises, hypothesis, kb=None, **kwargs).
pass


**Hint:** Return `NLILabel.NEUTRAL` for the dog example without KB, and `NLILabel.ENTAILMENT` when the KB contains `isa_wn(dog, animal)` or `isa_wn(run, move)`.


In [21]:
# Solution Exercise 7
def kb_contains(kb, needle):
    return any(needle == item for item in (kb or []))


def mock_langpro_api_call(premises, hypothesis, kb=None, **kwargs):
    kb = list(kb or [])
    premise_text = " ".join(premises).lower()
    hypothesis_text = hypothesis.lower()

    if "woman is smiling" in premise_text and "woman is smiling" in hypothesis_text:
        label = NLILabel.ENTAILMENT
    elif "cat is sleeping" in premise_text and "dog is running" in hypothesis_text:
        label = NLILabel.CONTRADICTION
    elif "dog is running" in premise_text and "animal is moving" in hypothesis_text:
        if kb_contains(kb, "isa_wn(dog, animal)") or kb_contains(kb, "isa_wn(run, move)"):
            label = NLILabel.ENTAILMENT
        else:
            label = NLILabel.NEUTRAL
    elif "chef is slicing" in premise_text and "someone is cutting" in hypothesis_text:
        label = NLILabel.NEUTRAL
    else:
        label = NLILabel.NEUTRAL

    return LangProResult(
        label=label,
        kb=kb,
        proofs={"mock": {"info": ["closed"] if label != NLILabel.NEUTRAL else ["open"]}},
    )

baseline_probe = mock_langpro_api_call(selected_problem.premises, selected_problem.hypothesis)
kb_probe = mock_langpro_api_call(selected_problem.premises, selected_problem.hypothesis, kb=filtered_kb)

print("Baseline label:", baseline_probe.label.value)
print("With-KB label:", kb_probe.label.value)


Baseline label: neutral
With-KB label: entailment


In [22]:
# Check Exercise 7
assert baseline_probe.label == NLILabel.NEUTRAL
assert kb_probe.label == NLILabel.ENTAILMENT
print("Exercise 7 check passed.")


Exercise 7 check passed.


## Exercise 8 — Write a mock LLM and patch orchestration

**Learning goal:** Learn how to make the high-level orchestration pipeline testable offline.

**Assignment:** Define `mock_call_llm`, then patch `kbprojection.orchestration.langpro_api_call` and `kbprojection.orchestration.call_llm`.


In [23]:
# TODO Exercise 8:
# Define mock_call_llm(provider, model, prompt_style, prob, **kwargs)
# and patch orchestration.langpro_api_call / orchestration.call_llm.
pass


**Hint:** `process_single_problem` uses the functions imported into `kbprojection.orchestration`, so patch that module directly.


In [24]:
# Solution Exercise 8
def mock_call_llm(provider, model, prompt_style, prob, **kwargs):
    canned = {
        "fixed-dog": ["isa_wn(dog, animal)", "isa_wn(run, move)", "isa_wn(park, outdoors)"],
        "already-smile": [],
        "still-wrong-cat": ["isa_wn(cat, dog)"],
        "empty-filter-chef": ["isa_wn(chef, person)"],
    }
    return canned.get(prob.id, [])

orchestration.langpro_api_call = mock_langpro_api_call
orchestration.call_llm = mock_call_llm

print("Patched orchestration.langpro_api_call ->", orchestration.langpro_api_call.__name__)
print("Patched orchestration.call_llm ->", orchestration.call_llm.__name__)
print("Mock KB for fixed-dog:", mock_call_llm("mock", "mock-model", "icl", selected_problem))


Patched orchestration.langpro_api_call -> mock_langpro_api_call
Patched orchestration.call_llm -> mock_call_llm
Mock KB for fixed-dog: ['isa_wn(dog, animal)', 'isa_wn(run, move)', 'isa_wn(park, outdoors)']


In [25]:
# Check Exercise 8
assert orchestration.langpro_api_call is mock_langpro_api_call
assert orchestration.call_llm is mock_call_llm
assert "isa_wn(dog, animal)" in mock_call_llm("mock", "mock-model", "icl", selected_problem)
print("Exercise 8 check passed.")


Exercise 8 check passed.


## Exercise 9 — Process one problem end to end

**Learning goal:** Run the real orchestration logic with offline mocks.

**Assignment:** Use `process_single_problem` on the `fixed-dog` problem and inspect the baseline prediction, raw KB, filtered KB, and final status.


In [26]:
# TODO Exercise 9:
# Call process_single_problem(selected_problem, config=ex1_config).
pass


**Hint:** Because Exercise 8 patched orchestration, this call should not contact a real LLM or LangPro endpoint.


In [27]:
# Solution Exercise 9
single_result = orchestration.process_single_problem(selected_problem, config=ex1_config)

print("Problem:", single_result.problem.id)
print("Gold:", single_result.problem.gold_label.value)
print("No-KB prediction:", single_result.pred_no_kb.value if single_result.pred_no_kb else None)
print("Raw KB:", single_result.kb_raw)
print("Filtered KB:", single_result.kb_filtered)
print("With-KB prediction:", single_result.pred_with_kb.value if single_result.pred_with_kb else None)
print("Final status:", single_result.final_status.value)
print("Fixed by:", single_result.fixed_by)


Problem: fixed-dog
Gold: entailment
No-KB prediction: neutral
Raw KB: ['isa_wn(dog, animal)', 'isa_wn(run, move)', 'isa_wn(park, outdoors)']
Filtered KB: ['isa_wn(dog, animal)', 'isa_wn(run, move)', 'isa_wn(park, outdoors)', 'isa_wn(park, outdoor)']
With-KB prediction: entailment
Final status: fixed
Fixed by: both


In [28]:
# Check Exercise 9
assert single_result.final_status == ExperimentStatus.FIXED
assert single_result.pred_no_kb == NLILabel.NEUTRAL
assert single_result.pred_with_kb == NLILabel.ENTAILMENT
print("Exercise 9 check passed.")


Exercise 9 check passed.


## Exercise 10 — Run a small batch experiment

**Learning goal:** Use `process_kb_examples` to process multiple dataset examples.

**Assignment:** Run the mini dataset through the offline pipeline and collect all four outcomes.


In [29]:
# TODO Exercise 10:
# Use process_kb_examples(..., cache_dir=None) to process the tiny dataset.
pass


**Hint:** Use `max_matches=4`, `max_checked=4`, and `label_filter={"entailment"}`. `cache_dir=None` keeps this exercise from writing files.


In [30]:
# Solution Exercise 10
batch_config = ProblemConfig(
    llm_provider="mock",
    model="mock-model",
    prompt_style="icl",
    test_mode=TestMode.BOTH,
    run_ablation=False,
    verbose=False,
)

batch_results = list(
    orchestration.process_kb_examples(
        dataset=mini_loader,
        config=batch_config,
        split="dev",
        label_filter={"entailment"},
        max_matches=4,
        max_checked=4,
        problem_ids=None,
        cache_dir=None,
    )
)

for result in batch_results:
    print(result.problem.id, "->", result.final_status.value, "| fixed_by=", result.fixed_by)


[kb-processor] Processing split='dev', labels={'entailment'}
[kb-processor] Config: mode=both, ablation=False
[kb-processor] Using random sampling.

[kb-processor] #1 | Key: fixed-dog | Gold: entailment

[kb-processor] #2 | Key: already-smile | Gold: entailment

[kb-processor] #3 | Key: still-wrong-cat | Gold: entailment

[kb-processor] #4 | Key: empty-filter-chef | Gold: entailment
[kb-processor] Reached target of 4 results. Stopping.
fixed-dog -> fixed | fixed_by= both
already-smile -> already_correct | fixed_by= None
still-wrong-cat -> still_wrong | fixed_by= None
empty-filter-chef -> empty_kb_after_filter | fixed_by= None


In [31]:
# Check Exercise 10
status_values = {result.final_status.value for result in batch_results}
assert "fixed" in status_values
assert "already_correct" in status_values
assert "still_wrong" in status_values
assert "empty_kb_after_filter" in status_values
print("Exercise 10 check passed with statuses:", sorted(status_values))


Exercise 10 check passed with statuses: ['already_correct', 'empty_kb_after_filter', 'fixed', 'still_wrong']


## Exercise 11 — Summarize and optionally export results

**Learning goal:** Turn `ExperimentResult` objects into an analysis-friendly table.

**Assignment:** Write `summarize_results`, print compact rows, and keep export disabled by default.


In [32]:
# TODO Exercise 11:
# Define summarize_results(results), then optionally export rows.
pass


**Hint:** A row should include problem ID, gold label, no-KB prediction, with-KB prediction, final status, fixed_by, and KB strings.


In [33]:
# Solution Exercise 11
def enum_value(value):
    return value.value if hasattr(value, "value") else value


def summarize_results(results):
    rows = []
    for result in results:
        rows.append({
            "problem_id": result.problem.id,
            "gold_label": enum_value(result.problem.gold_label),
            "pred_no_kb": enum_value(result.pred_no_kb) if result.pred_no_kb else None,
            "pred_with_raw_kb": enum_value(result.pred_with_raw_kb) if result.pred_with_raw_kb else None,
            "pred_with_filtered_kb": enum_value(result.pred_with_kb) if result.pred_with_kb else None,
            "final_status": enum_value(result.final_status),
            "fixed_by": result.fixed_by,
            "kb_raw": " | ".join(result.kb_raw or []),
            "kb_filtered": " | ".join(result.kb_filtered or []),
        })
    return rows

summary_rows = summarize_results(batch_results)
print("Status counts:")
pprint(Counter(row["final_status"] for row in summary_rows))
print("\nRows:")
pprint(summary_rows)

RUN_EXPORT = False
if RUN_EXPORT:
    results_dir = Path("exercise_results")
    results_dir.mkdir(parents=True, exist_ok=True)
    csv_path = results_dir / "mini_experiment_summary.csv"
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
        writer.writeheader()
        writer.writerows(summary_rows)
    print("Wrote", csv_path.resolve())
else:
    print("\nExport skipped. Set RUN_EXPORT = True to write exercise_results/mini_experiment_summary.csv")


Status counts:
Counter({'fixed': 1,
         'already_correct': 1,
         'still_wrong': 1,
         'empty_kb_after_filter': 1})

Rows:
[{'final_status': 'fixed',
  'fixed_by': 'both',
  'gold_label': 'entailment',
  'kb_filtered': 'isa_wn(dog, animal) | isa_wn(run, move) | isa_wn(park, '
                 'outdoors) | isa_wn(park, outdoor)',
  'kb_raw': 'isa_wn(dog, animal) | isa_wn(run, move) | isa_wn(park, outdoors)',
  'pred_no_kb': 'neutral',
  'pred_with_filtered_kb': 'entailment',
  'pred_with_raw_kb': 'entailment',
  'problem_id': 'fixed-dog'},
 {'final_status': 'already_correct',
  'fixed_by': None,
  'gold_label': 'entailment',
  'kb_filtered': '',
  'kb_raw': '',
  'pred_no_kb': 'entailment',
  'pred_with_filtered_kb': None,
  'pred_with_raw_kb': None,
  'problem_id': 'already-smile'},
 {'final_status': 'still_wrong',
  'fixed_by': None,
  'gold_label': 'entailment',
  'kb_filtered': '',
  'kb_raw': '',
  'pred_no_kb': 'contradiction',
  'pred_with_filtered_kb': None,
  'p

In [34]:
# Check Exercise 11
assert len(summary_rows) == 4
assert {"problem_id", "final_status", "kb_filtered"}.issubset(summary_rows[0])
assert not Path("exercise_results").exists() or not RUN_EXPORT
print("Exercise 11 check passed.")


Exercise 11 check passed.


## Exercise 12 — Inspect failure modes

**Learning goal:** Learn to debug experiment outcomes instead of only counting successes.

**Assignment:** Group results by `final_status`, then inspect the examples that did not become `fixed` or `already_correct`.


In [35]:
# TODO Exercise 12:
# Group batch_results by final_status and inspect non-success cases.
pass


**Hint:** The useful failure fields are usually `pred_no_kb`, `kb_raw`, `kb_filtered`, and `final_status`.


In [36]:
# Solution Exercise 12
results_by_status = {}
for result in batch_results:
    results_by_status.setdefault(result.final_status.value, []).append(result)

print("Results by status:")
for status, items in results_by_status.items():
    print(f"- {status}: {[item.problem.id for item in items]}")

print("\nFailure-analysis examples:")
for result in batch_results:
    if result.final_status in {ExperimentStatus.FIXED, ExperimentStatus.ALREADY_CORRECT}:
        continue
    print("=" * 80)
    print("ID:", result.problem.id)
    print("Premise:", result.problem.premises)
    print("Hypothesis:", result.problem.hypothesis)
    print("No-KB prediction:", enum_value(result.pred_no_kb) if result.pred_no_kb else None)
    print("Raw KB:", result.kb_raw)
    print("Filtered KB:", result.kb_filtered)
    print("Final status:", result.final_status.value)


Results by status:
- fixed: ['fixed-dog']
- already_correct: ['already-smile']
- still_wrong: ['still-wrong-cat']
- empty_kb_after_filter: ['empty-filter-chef']

Failure-analysis examples:
ID: still-wrong-cat
Premise: ['A cat is sleeping on the sofa.']
Hypothesis: A dog is running outside.
No-KB prediction: contradiction
Raw KB: None
Filtered KB: None
Final status: still_wrong
ID: empty-filter-chef
Premise: ['A chef is slicing a tomato.']
Hypothesis: Someone is cutting a tomato.
No-KB prediction: neutral
Raw KB: ['isa_wn(chef, person)']
Filtered KB: []
Final status: empty_kb_after_filter


In [37]:
# Check Exercise 12
assert "still_wrong" in results_by_status
assert "empty_kb_after_filter" in results_by_status
print("Exercise 12 check passed.")


Exercise 12 check passed.


## Exercise 13 — Optional live LangPro endpoint diagnostic

**Learning goal:** Understand how to test the live LangPro JSON endpoint separately from the experiment loop.

**Assignment:** Keep the live call disabled by default. Read the diagnostic function and expected failure. If you later have network access and want to test the stale endpoint, set `RUN_LIVE_LANGPRO_DIAGNOSTIC = True`.


In [38]:
# TODO Exercise 13:
# Read the diagnostic function in the solution cell.
# Do not enable the live call unless you really want to test the external endpoint.
pass


**Hint:** The current repository default endpoint is `https://langpro-annotator.hum.uu.nl/langpro-api/prove/`. In recent testing it returned HTML `404 Not Found`, not JSON, which makes `langpro_api_call` return label `-` with a JSON decode error.


In [39]:
# Solution Exercise 13
RUN_LIVE_LANGPRO_DIAGNOSTIC = False
LANGPRO_JSON_ENDPOINT = "https://langpro-annotator.hum.uu.nl/langpro-api/prove/"


def diagnose_live_langpro_endpoint():
    import requests

    payload = {
        "premises": ["The milk is being drunk by a cat"],
        "hypothesis": "The cat is drinking some milk",
        "prover_config": ["aall", "allInt"],
        "parser": "easyccg",
        "ral": 200,
        "kb": [],
        "senses": "all",
    }
    response = requests.post(LANGPRO_JSON_ENDPOINT, json=payload, timeout=30)
    print("status:", response.status_code)
    print("content-type:", response.headers.get("content-type"))
    print("body prefix:", repr(response.text[:300]))
    try:
        data = response.json()
        print("JSON keys:", sorted(data.keys()) if isinstance(data, dict) else type(data).__name__)
    except Exception as exc:
        print("JSON parse failed:", type(exc).__name__, exc)

if RUN_LIVE_LANGPRO_DIAGNOSTIC:
    diagnose_live_langpro_endpoint()
else:
    print("Live diagnostic skipped.")
    print("Expected current failure if enabled: HTTP 404 HTML response, followed by JSONDecodeError.")


Live diagnostic skipped.
Expected current failure if enabled: HTTP 404 HTML response, followed by JSONDecodeError.


In [40]:
# Check Exercise 13
assert RUN_LIVE_LANGPRO_DIAGNOSTIC is False
assert LANGPRO_JSON_ENDPOINT.endswith("/langpro-api/prove/")
print("Exercise 13 check passed without making a live network call.")


Exercise 13 check passed without making a live network call.


# Wrap-up assignment

You now have the basic experiment workflow:

1. Represent examples with `NLIProblem`.
2. Load or iterate examples with a dataset loader.
3. Build prompts and parse LLM KB output.
4. Filter KB injections against the premise/hypothesis.
5. Use `process_single_problem` and `process_kb_examples` for orchestration.
6. Summarize outcomes and inspect failures.

Suggested extension: add a fifth mini problem where raw KB fixes the example but filtering removes the helpful relation. Then rerun Exercises 10-12 and explain what changed.
